# First 1D-CNN Pilot

This notebook trains the first deep-learning model on the harmonized representation selected by the cross-dataset ablation: acceleration magnitudes from the lower-back, left-foot, and right-foot placements.

Architecture: three temporal convolution blocks, batch normalization, GELU activation, max pooling, global average pooling, dropout, and a binary output. The pilot uses participant-level splits, participant-balanced sample weights, fold-specific normalization, and participant-level validation aggregation.

The default run is fold 0 only. After the loop is verified, change RUN_FOLDS to all five folds for the proper development estimate.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RAW_ARRAY_PATH = PROCESSED / 'validated_gait_windows_float32.npy'
MAG_ARRAY_PATH = PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'
SPLITS_PATH = PROJECT_ROOT / 'data' / 'interim' / 'participant_splits.csv'
RUN_FOLDS = [0, 1, 2, 3, 4]
EPOCHS = 6
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Torch:', torch.__version__)
print('Device:', DEVICE)


Torch: 2.13.0+cpu
Device: cpu


In [2]:
metadata = pd.read_csv(METADATA_PATH)
splits = pd.read_csv(SPLITS_PATH)
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
raw_windows = np.load(RAW_ARRAY_PATH, mmap_mode='r')
if not MAG_ARRAY_PATH.exists():
    magnitude_windows = np.lib.format.open_memmap(MAG_ARRAY_PATH, mode='w+', dtype='float32', shape=(len(raw_windows), raw_windows.shape[1], 3))
    for start in range(0, len(raw_windows), 512):
        batch = np.asarray(raw_windows[start:start + 512], dtype=np.float32)
        magnitude_windows[start:start + len(batch), :, 0] = np.linalg.norm(batch[:, :, 0:3], axis=2)
        magnitude_windows[start:start + len(batch), :, 1] = np.linalg.norm(batch[:, :, 6:9], axis=2)
        magnitude_windows[start:start + len(batch), :, 2] = np.linalg.norm(batch[:, :, 12:15], axis=2)
    magnitude_windows.flush()
magnitude_windows = np.load(MAG_ARRAY_PATH, mmap_mode='r')
assert magnitude_windows.shape == (len(metadata), 500, 3)
assert np.isfinite(np.asarray(magnitude_windows[:16])).all()
print('Magnitude array:', magnitude_windows.shape)


Magnitude array: (18511, 500, 3)


In [3]:
def fold_roles(fold):
    role_map = splits[splits['fold'].eq(fold)].set_index('participant_key')['role']
    return metadata['participant_key'].map(role_map)


def fold_normalization(train_indices):
    total = np.zeros(3, dtype=np.float64)
    total_sq = np.zeros(3, dtype=np.float64)
    count = 0
    for start in range(0, len(train_indices), 512):
        batch = np.asarray(magnitude_windows[train_indices[start:start + 512]], dtype=np.float32)
        total += batch.sum(axis=(0, 1))
        total_sq += np.square(batch).sum(axis=(0, 1))
        count += batch.shape[0] * batch.shape[1]
    mean = total / count
    std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-8))
    return mean.astype(np.float32), std.astype(np.float32)


class MagnitudeDataset(Dataset):
    def __init__(self, indices, mean, std, weights=None):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.mean = mean.reshape(1, 3)
        self.std = std.reshape(1, 3)
        self.weights = np.ones(len(self.indices), dtype=np.float32) if weights is None else np.asarray(weights, dtype=np.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        source_index = self.indices[item]
        signal = np.asarray(magnitude_windows[source_index], dtype=np.float32)
        signal = ((signal - self.mean) / self.std).T.copy()
        label = np.float32(metadata.iloc[source_index]['label_binary'])
        return torch.from_numpy(signal), torch.tensor(label), torch.tensor(self.weights[item]), torch.tensor(source_index)


class GaitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=9, padding=4),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(128, 1))

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


In [4]:
def evaluate(model, loader):
    model.eval()
    records = []
    with torch.no_grad():
        for signals, labels, _, indices in loader:
            probabilities = torch.sigmoid(model(signals.to(DEVICE))).cpu().numpy()
            for index, label, probability in zip(indices.numpy(), labels.numpy(), probabilities):
                records.append({
                    'window_index': int(index),
                    'participant_key': metadata.iloc[int(index)]['participant_key'],
                    'dataset_id': metadata.iloc[int(index)]['dataset_id'],
                    'label_binary': int(label),
                    'probability': float(probability),
                })
    frame = pd.DataFrame(records)
    participant = frame.groupby(['participant_key', 'dataset_id', 'label_binary'], as_index=False)['probability'].mean()
    y_true = participant['label_binary'].to_numpy()
    y_prob = participant['probability'].to_numpy()
    metrics = {
        'participants': len(participant),
        'balanced_accuracy': balanced_accuracy_score(y_true, (y_prob >= 0.5).astype(int)),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'f1': f1_score(y_true, (y_prob >= 0.5).astype(int)),
    }
    return metrics, participant


fold_results = []
all_predictions = []
for fold in RUN_FOLDS:
    roles = fold_roles(fold)
    train_indices = np.flatnonzero(roles.eq('training').to_numpy())
    validation_indices = np.flatnonzero(roles.eq('validation').to_numpy())
    mean, std = fold_normalization(train_indices)
    counts = metadata.iloc[train_indices].groupby('participant_key').size()
    participant_weights = metadata.iloc[train_indices]['participant_key'].map(1.0 / counts).to_numpy()
    class_counts = metadata.iloc[train_indices].groupby('label_binary').size()
    class_weights = metadata.iloc[train_indices]['label_binary'].map(len(train_indices) / (2.0 * class_counts)).to_numpy()
    weights = participant_weights * class_weights
    weights = weights / weights.mean()

    train_loader = DataLoader(MagnitudeDataset(train_indices, mean, std, weights), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    validation_loader = DataLoader(MagnitudeDataset(validation_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = GaitCNN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    best_auc = -np.inf
    best_state = None
    patience_left = 2

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for signals, labels, batch_weights, _ in train_loader:
            optimizer.zero_grad()
            logits = model(signals.to(DEVICE))
            loss = (nn.functional.binary_cross_entropy_with_logits(logits, labels.to(DEVICE), reduction='none') * batch_weights.to(DEVICE)).mean()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))
        metrics, participant_predictions = evaluate(model, validation_loader)
        print('fold={} epoch={} loss={:.4f} auc={:.3f} bal_acc={:.3f}'.format(fold, epoch, np.mean(losses), metrics['roc_auc'], metrics['balanced_accuracy']))
        if metrics['roc_auc'] > best_auc:
            best_auc = metrics['roc_auc']
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_left = 2
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    model.load_state_dict(best_state)
    metrics, participant_predictions = evaluate(model, validation_loader)
    metrics['fold'] = fold
    fold_results.append(metrics)
    participant_predictions['fold'] = fold
    all_predictions.append(participant_predictions)
    torch.save({'model_state_dict': model.state_dict(), 'mean': mean, 'std': std, 'fold': fold}, PROCESSED / f'cnn_magnitude_fold_{fold}_best.pt')

fold_results = pd.DataFrame(fold_results)
all_predictions = pd.concat(all_predictions, ignore_index=True)
print(fold_results.round(3).to_string(index=False))
fold_results.to_csv(PROCESSED / 'cnn_pilot_fold_results.csv', index=False)
all_predictions.to_csv(PROCESSED / 'cnn_pilot_participant_predictions.csv', index=False)


fold=0 epoch=1 loss=0.3380 auc=0.931 bal_acc=0.792


fold=0 epoch=2 loss=0.2452 auc=0.898 bal_acc=0.671


fold=0 epoch=3 loss=0.2394 auc=0.921 bal_acc=0.716


fold=1 epoch=1 loss=0.3309 auc=0.929 bal_acc=0.840


fold=1 epoch=2 loss=0.2634 auc=0.938 bal_acc=0.872


...

fold=4 epoch=3 loss=0.2400 auc=0.986 bal_acc=0.933


fold=4 epoch=4 loss=0.2164 auc=0.993 bal_acc=0.948


fold=4 epoch=5 loss=0.1904 auc=0.993 bal_acc=0.829


fold=4 epoch=6 loss=0.1985 auc=0.993 bal_acc=0.800


 participants  balanced_accuracy  roc_auc    f1  fold
           57              0.792    0.931 0.737     0
           58              0.872    0.938 0.921     1
           57              0.861    0.981 0.839     2
           56              0.933    0.978 0.941     3
           56              0.948    0.993 0.957     4
